# CEVAE実験 v3：潜在変数 Z が2次元の場合

v2（Z が1次元）を拡張し、**Z ∈ R²** にした DGP で X-learner と CEVAE を比較する。

- X-learner・CEVAE ともに data_seed=0 でハイパラ探索 → 全 seed に流用
- 上から順に実行するだけで完結（約10分）

In [ ]:
import warnings
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import uniform, randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from xgboost import XGBRegressor
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_toy_v3(N=20000, d=8, sigma_x=3.5, sigma_y=1.0, seed=0):
    """v3 DGP: Z が2次元。
    Z = [Z1, Z2] ~ N(0, I2)
    X = tanh(Z @ A) + noise,  A ∈ R^{2×d} ~ N(1,1)
    T | Z ~ Bernoulli(σ(1.5*Z1 + 0.5*Z2))
    τ(Z) = 1.0 + Z1 + 0.5*Z2 + 0.8*(Z1²+Z2²-2)
    Y0 = 2.0*Z1 + 1.0*Z2 + noise
    Y1 = Y0 + τ(Z)
    """
    rng = np.random.default_rng(seed)
    Z = rng.normal(0, 1, size=(N, 2))
    A = rng.normal(1.0, 1.0, size=(2, d))
    X = np.tanh(Z @ A) + rng.normal(0, sigma_x, size=(N, d))
    w_T = np.array([1.5, 0.5])
    T = rng.binomial(1, sigmoid(Z @ w_T))
    w_tau = np.array([1.0, 0.5])
    tau = 1.0 + Z @ w_tau + 0.8 * (np.sum(Z ** 2, axis=1) - 2)
    w_Y = np.array([2.0, 1.0])
    base = Z @ w_Y
    Y0 = base + rng.normal(0, sigma_y, size=N)
    Y1 = base + tau + rng.normal(0, sigma_y, size=N)
    Y = np.where(T == 1, Y1, Y0)
    return X, T, Y, Y0, Y1, tau, Z


In [ ]:
DATA_SEED = 0
X, T, Y, Y0, Y1, tau, Z = generate_toy_v3(seed=DATA_SEED)
d = X.shape[1]
N = len(Y)

proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T; df_full["Y"] = Y; df_full["tau"] = tau

# 訓練/テスト分割（全実験共通）
df_train, df_test = train_test_split(df_full, test_size=0.3,
                                      random_state=DATA_SEED, stratify=df_full["T"])

print(f"N={N}, d={d}, train={len(df_train)}, test={len(df_test)}")
print(f"真のATE       : {tau.mean():.3f}")
print(f"ナイーブATE   : {Y[T==1].mean()-Y[T==0].mean():.3f}")
print(f"corr(T, Z1)   : {np.corrcoef(T, Z[:,0])[0,1]:.3f}")
print(f"corr(T, Z2)   : {np.corrcoef(T, Z[:,1])[0,1]:.3f}")


## X-learner ハイパラ探索（data_seed=0）

In [ ]:
XGB_PARAM_DIST = {
    "n_estimators": randint(300, 2000),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(2, 8),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 10.0),
}

def search_xgb(X_in, y_in, n_iter=30, cv=3, seed=0):
    s = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=seed, verbosity=0),
        param_distributions=XGB_PARAM_DIST,
        n_iter=n_iter, cv=cv, scoring="neg_mean_squared_error",
        n_jobs=-1, random_state=seed,
    )
    s.fit(X_in, y_in)
    return s.best_estimator_

def run_xlearner(df_tr, df_te, proxy_cols, seed=0):
    df0 = df_tr[df_tr["T"] == 0]
    df1 = df_tr[df_tr["T"] == 1]
    m0 = search_xgb(df0[proxy_cols], df0["Y"], seed=seed)
    m1 = search_xgb(df1[proxy_cols], df1["Y"], seed=seed)
    D0 = m1.predict(df0[proxy_cols]) - df0["Y"].values
    D1 = df1["Y"].values - m0.predict(df1[proxy_cols])
    x0 = search_xgb(df0[proxy_cols], D0, seed=seed)
    x1 = search_xgb(df1[proxy_cols], D1, seed=seed)
    prop = LogisticRegression(max_iter=1000).fit(df_tr[proxy_cols], df_tr["T"])
    g0 = prop.predict_proba(df_te[proxy_cols])[:, 0]
    tau_pred = g0 * x0.predict(df_te[proxy_cols]) + (1 - g0) * x1.predict(df_te[proxy_cols])
    return tau_pred, (m0, m1, x0, x1, prop)

print("X-learner ハイパラ探索中（data_seed=0）...")
xl_tau_pred, xl_models_fixed = run_xlearner(df_train, df_test, proxy_cols, seed=0)
tau_true_test_s0 = df_test["tau"].values
xl_pehe_s0 = np.sqrt(np.mean((tau_true_test_s0 - xl_tau_pred) ** 2))
print(f"X-learner √PEHE (data_seed=0): {xl_pehe_s0:.3f}")


## ネットワーク・関数定義

In [ ]:
class DecoderX(nn.Module):
    def __init__(self, z_dim, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, z): return self.net(z)

class DecoderT(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.net(z).squeeze(-1)

class DecoderY(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.head0(z).squeeze(-1), self.head1(z).squeeze(-1)

class EncoderT(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class EncoderY(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.head0(x).squeeze(-1), self.head1(x).squeeze(-1)

class EncoderZ(nn.Module):
    def __init__(self, d, z_dim, hidden):
        super().__init__()
        self.trunk   = nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
        self.head0   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.head1   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.mu0     = nn.Linear(hidden, z_dim)
        self.logvar0 = nn.Linear(hidden, z_dim)
        self.mu1     = nn.Linear(hidden, z_dim)
        self.logvar1 = nn.Linear(hidden, z_dim)
    def forward(self, x, t, y):
        h  = self.trunk(x)
        hy = torch.cat([h, y.unsqueeze(-1)], dim=-1)
        h0, h1 = self.head0(hy), self.head1(hy)
        mu0, lv0 = self.mu0(h0), self.logvar0(h0)
        mu1, lv1 = self.mu1(h1), self.logvar1(h1)
        tc = t.unsqueeze(-1)
        return torch.where(tc==1, mu1, mu0), torch.where(tc==1, lv1, lv0)

def reparameterize(mu, logvar):
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

def kl_to_standard_normal(mu, logvar):
    return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).sum(dim=-1).mean()

def linear_kl_anneal(step, total_steps):
    return 1.0 if total_steps <= 0 else min(1.0, step / total_steps)

@dataclass
class CEVAEConfig:
    z_dim: int = 2
    hidden: int = 64
    beta_kl_max: float = 0.3
    kl_anneal_epochs: int = 100
    w_recon_y: float = 1.0
    w_aux_y: float = 1.0
    epochs: int = 200
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 0

def build_cevae(cfg, d):
    torch.manual_seed(cfg.seed)
    return {
        "decoder_x": DecoderX(cfg.z_dim, d, cfg.hidden),
        "decoder_t": DecoderT(cfg.z_dim, cfg.hidden),
        "decoder_y": DecoderY(cfg.z_dim, cfg.hidden),
        "encoder_t": EncoderT(d, cfg.hidden),
        "encoder_y": EncoderY(d, cfg.hidden),
        "encoder_z": EncoderZ(d, cfg.z_dim, cfg.hidden),
    }

def cevae_loss_fn(nets, x, t, y, beta_kl, w_recon_y=1.0, w_aux_y=1.0):
    mu_z, logvar_z = nets["encoder_z"](x, t, y)
    z = reparameterize(mu_z, logvar_z)
    x_hat = nets["decoder_x"](z)
    t_logit_p = nets["decoder_t"](z)
    y0p, y1p = nets["decoder_y"](z)
    y_hat_p = torch.where(t == 1, y1p, y0p)
    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)
    kl = kl_to_standard_normal(mu_z, logvar_z)
    t_logit_q = nets["encoder_t"](x)
    y0q, y1q = nets["encoder_y"](x)
    y_hat_q = torch.where(t == 1, y1q, y0q)
    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)
    return recon_x + recon_t + w_recon_y*recon_y + beta_kl*kl + aux_t + w_aux_y*aux_y

def train_cevae(nets, cfg, x_tr, t_tr, y_tr):
    torch.manual_seed(cfg.seed)
    all_params = [p for net in nets.values() for p in net.parameters()]
    opt = torch.optim.Adam(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    n = x_tr.shape[0]
    n_batches = max(1, n // cfg.batch_size)
    for epoch in range(cfg.epochs):
        beta_kl = cfg.beta_kl_max * linear_kl_anneal(epoch, cfg.kl_anneal_epochs)
        perm = torch.randperm(n)
        for b in range(n_batches):
            idx = perm[b*cfg.batch_size:(b+1)*cfg.batch_size]
            opt.zero_grad()
            cevae_loss_fn(nets, x_tr[idx], t_tr[idx], y_tr[idx],
                          beta_kl, cfg.w_recon_y, cfg.w_aux_y).backward()
            opt.step()

def compute_val_elbo(nets, x_val, t_val, y_val):
    with torch.no_grad():
        return cevae_loss_fn(nets, x_val, t_val, y_val, beta_kl=1.0).item()

def estimate_ite(nets, x):
    with torch.no_grad():
        t_hat = (torch.sigmoid(nets["encoder_t"](x)) > 0.5).float()
        y0q, y1q = nets["encoder_y"](x)
        y_hat = torch.where(t_hat == 1, y1q, y0q)
        mu_z, _ = nets["encoder_z"](x, t_hat, y_hat)
        y0p, y1p = nets["decoder_y"](mu_z)
        return (y1p - y0p).numpy()

def z_corr_general(nets, x, t, y, Z_true):
    """z_dim が何次元でも使える z 回収スコア。
    真の Z の各次元に対して最も相関の高い mu_z 次元を選び、平均絶対相関を返す。
    """
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x, t, y)
    mz = mu_z.numpy()
    if Z_true.ndim == 1:
        Z_true = Z_true[:, None]
    d_true = Z_true.shape[1]
    z_dim  = mz.shape[1]
    scores = []
    for j in range(d_true):
        best = max(abs(np.corrcoef(mz[:, k], Z_true[:, j])[0, 1]) for k in range(z_dim))
        scores.append(best)
    return float(np.mean(scores))

## CEVAE ハイパラ探索（data_seed=0, バリデーション ELBO 基準）

トレーニングデータを inner_train (80%) / val (20%) に分けて探索し、
バリデーション ELBO が最小の設定を全 seed に流用する。

In [ ]:
# inner_train / val 分割（テストは触らない）
df_inner, df_val = train_test_split(df_train, test_size=0.2,
                                     random_state=0, stratify=df_train["T"])

def df_to_tensors(df):
    x = torch.tensor(df[proxy_cols].values, dtype=torch.float32)
    t = torch.tensor(df["T"].values, dtype=torch.float32)
    y = torch.tensor(df["Y"].values, dtype=torch.float32)
    return x, t, y

x_inner, t_inner, y_inner = df_to_tensors(df_inner)
x_val,   t_val,   y_val   = df_to_tensors(df_val)

SEARCH_SPACE = {
    "z_dim":            [1, 2, 3],
    "hidden":           [32, 64, 128],
    "beta_kl_max":      [0.1, 0.3, 0.5, 1.0],
    "kl_anneal_epochs": [50, 100, 150],
    "epochs":           [150, 300],
    "weight_decay":     [0.0, 1e-4, 1e-3],
}
N_ITER = 20

rng_search = random.Random(42)
search_records = []

print(f"CEVAE ランダムサーチ（n_iter={N_ITER}）...")
for i in range(N_ITER):
    cfg = CEVAEConfig(
        z_dim=rng_search.choice(SEARCH_SPACE["z_dim"]),
        hidden=rng_search.choice(SEARCH_SPACE["hidden"]),
        beta_kl_max=rng_search.choice(SEARCH_SPACE["beta_kl_max"]),
        kl_anneal_epochs=rng_search.choice(SEARCH_SPACE["kl_anneal_epochs"]),
        epochs=rng_search.choice(SEARCH_SPACE["epochs"]),
        weight_decay=rng_search.choice(SEARCH_SPACE["weight_decay"]),
        seed=0,
    )
    nets = build_cevae(cfg, d)
    train_cevae(nets, cfg, x_inner, t_inner, y_inner)
    val_elbo = compute_val_elbo(nets, x_val, t_val, y_val)
    search_records.append({"iter": i, "val_elbo": val_elbo,
                           "z_dim": cfg.z_dim, "hidden": cfg.hidden,
                           "beta_kl_max": cfg.beta_kl_max,
                           "kl_anneal_epochs": cfg.kl_anneal_epochs,
                           "epochs": cfg.epochs, "weight_decay": cfg.weight_decay})
    print(f"  [{i+1:2d}/{N_ITER}] val_elbo={val_elbo:.3f}  "
          f"z={cfg.z_dim} h={cfg.hidden} β={cfg.beta_kl_max} "
          f"anneal={cfg.kl_anneal_epochs} ep={cfg.epochs} wd={cfg.weight_decay}")

df_search = pd.DataFrame(search_records).sort_values("val_elbo")
best_row = df_search.iloc[0]
print(f"\n=== ベスト設定（上位5件）===")
print(df_search[["val_elbo","z_dim","hidden","beta_kl_max","kl_anneal_epochs","epochs","weight_decay"]].head(5).to_string(index=False))

BEST_CEVAE_CFG = CEVAEConfig(
    z_dim=int(best_row["z_dim"]),
    hidden=int(best_row["hidden"]),
    beta_kl_max=float(best_row["beta_kl_max"]),
    kl_anneal_epochs=int(best_row["kl_anneal_epochs"]),
    epochs=int(best_row["epochs"]),
    weight_decay=float(best_row["weight_decay"]),
    seed=0,
)
print(f"\nBEST_CEVAE_CFG: {BEST_CEVAE_CFG}")

## マルチシード実験（data_seed 0〜9）

両手法とも data_seed=0 で探索したパラメータを全 seed に流用。

In [ ]:
N_DATA_SEEDS = 10
records = []

for ds in range(N_DATA_SEEDS):
    print(f"data_seed={ds} ...", end=" ", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_v3(seed=ds)

    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts; df_s["Y"] = Ys; df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3,
                                         random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]

    # ── X-learner（固定パラメータで再学習）────────────────────────────────
    m0_f, m1_f, x0_f, x1_f, _ = xl_models_fixed
    df0s = df_tr_s[df_tr_s["T"]==0]; df1s = df_tr_s[df_tr_s["T"]==1]
    m0s = m0_f.__class__(**m0_f.get_params()).fit(df0s[proxy_cols], df0s["Y"])
    m1s = m1_f.__class__(**m1_f.get_params()).fit(df1s[proxy_cols], df1s["Y"])
    D0s = m1s.predict(df0s[proxy_cols]) - df0s["Y"].values
    D1s = df1s["Y"].values - m0s.predict(df1s[proxy_cols])
    x0s = x0_f.__class__(**x0_f.get_params()).fit(df0s[proxy_cols], D0s)
    x1s = x1_f.__class__(**x1_f.get_params()).fit(df1s[proxy_cols], D1s)
    props = LogisticRegression(max_iter=1000).fit(df_tr_s[proxy_cols], df_tr_s["T"])
    g0s = props.predict_proba(df_te_s[proxy_cols])[:, 0]
    tau_xl = g0s * x0s.predict(df_te_s[proxy_cols]) + (1-g0s) * x1s.predict(df_te_s[proxy_cols])
    xl_pehe_s = np.sqrt(np.mean((tau_true_s - tau_xl)**2))

    # ── CEVAE（ベスト設定で学習）──────────────────────────────────────────
    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    cfg_s = CEVAEConfig(**{k: getattr(BEST_CEVAE_CFG, k)
                           for k in BEST_CEVAE_CFG.__dataclass_fields__})
    nets_s = build_cevae(cfg_s, d)
    train_cevae(nets_s, cfg_s, xt[tr_idx], tt[tr_idx], yt[tr_idx])
    tau_cv = estimate_ite(nets_s, xt[te_idx])
    cv_pehe_s = np.sqrt(np.mean((tau_true_s - tau_cv)**2))
    zc_s = z_corr_general(nets_s, xt[te_idx], tt[te_idx], yt[te_idx], Z_te_s)

    records.append({"data_seed": ds, "xl_pehe": xl_pehe_s,
                    "cv_pehe": cv_pehe_s, "z_corr": zc_s})
    print(f"XL={xl_pehe_s:.3f}  CEVAE={cv_pehe_s:.3f}  z_corr={zc_s:.3f}")

df_res = pd.DataFrame(records)
print("\n=== 集計 ===")
print(df_res[["xl_pehe","cv_pehe","z_corr"]].agg(["mean","std"]).round(3))
print(f"\nCEVAE 勝利(seed別): {(df_res.cv_pehe < df_res.xl_pehe).sum()}/10 seed")